<a href="https://colab.research.google.com/github/takatakamanbou/MVA/blob/2025/MVA2025_ex10notebookC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVA2025 ex10notebookC

<img width=64 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/MVA/MVA-logo.png"> https://www-tlab.math.ryukoku.ac.jp/wiki/?MVA

----
## 確率密度推定の実験 / 確率密度推定の応用
---



In [ ]:
# 必要なパッケージのインポート
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

# NumPy の 疑似乱数生成器（rng = random number generator）
from numpy.random import default_rng
rng = default_rng() # 疑似乱数生成器を初期化

# SciPy のもろもろ
import scipy as sp
from scipy.spatial import distance
from scipy.stats import norm, multivariate_normal

# Python による「コンピュータビジョン(Computer Vision)」のためのライブラリ OpenCV のパッケージをインポート
import cv2

---
### $1.$ 確率密度推定の実験

notebookB では，実在の，正規分布から得られたかどうかが定かでないデータに正規分布を当てはめる（正規分布のパラメータを推定する）実験を行いました．
ここでは，疑似乱数によって理論的に正規分布に従うデータを生成し，そのパラメータを推定する実験を行ってみましょう．この場合，データの生成に用いたパラメータ（真のパラメータと呼ぶことにします）が分かっているので，推定されたパラメータの正しさを評価することができます．

推定に用いるデータのサンプルサイズ（データ数）の多い少ないは，推定の正しさにどのように影響するでしょうか？

#### 1次元正規分布の場合

In [ ]:
#@markdown サンプルサイズを指定して実行しよう．
N = 10  #@param [10,  100, 1000] {type: "raw"}

# 真の（母集団の）平均と分散
mu, sig2 = 3, 4

# 平均 mu, 分散 sig2 に従う正規乱数の標本を N 個抽出
X = np.sqrt(sig2) * rng.standard_normal(N) + mu

# 平均と分散を推定
mu_est   = np.mean(X)
sig2_est = np.var(X)

# 標本のヒストグラムと推定された正規分布を描く
xmin, xmax = mu - 5*np.sqrt(sig2), mu + 5*np.sqrt(sig2)
xx = np.linspace(xmin, xmax, num=100)
fig, ax = plt.subplots(1, figsize=(6, 4))

# ヒストグラム
ax.hist(X, density=True)

# 真の正規分布
px = norm.pdf(xx, loc=mu, scale=np.sqrt(sig2))
ax.plot(xx, px, linewidth=3, color='#0000ff', label='true')
ax.axvline(mu, color='#0000ff', linestyle='--')

# 推定された正規分布
px = norm.pdf(xx, loc=mu_est, scale=np.sqrt(sig2_est))
ax.plot(xx, px, linewidth=3, color='red', label='estimated')
ax.axvline(mu_est, color='#ff0000', linestyle='--')

#ax.axvline(0, color='gray')
ax.set_xlim(xmin, xmax)
ax.set_ylim(0, 0.4)
ax.legend()
plt.show()
print(f'N = {N}')
print(f'真の平均と分散: ({mu:.2f}, {sig2:.2f}) 推定された平均と分散: ({mu_est:.2f}, {sig2_est:.2f})')
print()

上のセルを実行すると，1次元正規分布に従う乱数の標本からその正規分布の平均と分散を最尤推定します．
青色が真の分布の確率密度関数を，赤色がデータから推定された分布の確率密度関数を表します．
`N`はサンプルサイズを表します．

(1) `N` を 10 としてセルを実行し，結果を観察しましょう．同じ `N` でも実行のたびに乱数値が変わるので，何度も実行し直して，グラフや推定される値が変化する様子を観察しましょう．

(2) `N` を 100 や 1000 にして (1) と同様のことをやりましょう．

推定の良さとサンプルサイズの間にはどのような関係がありそうでしょうか．サンプルサイズが大きい場合と小さい場合では，どちらのほうが真のパラメータ値に近い推定値が得られそうでしょうか．

次のセルを実行すると，グラフを描くのを省略して，上記の実験を10回行った結果を表示します．

In [ ]:
#@markdown サンプルサイズを指定して実行しよう．
N = 10  #@param [10,  100, 1000] {type: "raw"}

# 同じ実験を10回繰り返してみる
print(f'N = {N}  真の平均と分散: ({mu:.2f}, {sig2:.2f})')
for i in range(10):
    # 平均 mu, 分散 sig2 に従う正規乱数の標本を N 個抽出
    X = np.sqrt(sig2) * rng.standard_normal(N) + mu
    # 平均と分散を推定
    mu_est   = np.mean(X)
    sig2_est = np.var(X)
    print(f'実験{i+1}回目  推定された平均と分散: ({mu_est:.2f}, {sig2_est:.2f})')

サンプルサイズが大きい場合と小さい場合で，推定されるパラメータ値のばらつき方に変化はあるでしょうか．

#### 2次元正規分布の場合

2次元の場合についても同様の実験を行ってみましょう．

In [ ]:
#@markdown サンプルサイズを指定して実行しよう．
N = 20  #@param [20, 200, 2000] {type: "raw"}

# 真の（母集団の）平均と分散
mu  = np.array([7, 5])
cov = np.array([[6, -2],
                [-2, 3]])

# 平均 mu, 共分散 cov に従う正規乱数の標本を N 個抽出
X = rng.multivariate_normal(mu, cov, size=N)

# 平均と分散を推定
mu_est  = np.mean(X, axis=0)
XX = X - mu_est
cov_est = XX.T @ XX / N

# 散布図
fig, ax = plt.subplots(figsize=(5, 5))
xmin, xmax = -5, 15
ymin, ymax = -5, 15
ax.scatter(X[:, 0], X[:, 1], s=4)

# 真の正規分布
xx, yy = np.mgrid[xmin:xmax:0.1, ymin:ymax:0.1]
zz = multivariate_normal.pdf(np.dstack((xx, yy)), mu ,cov)
ax.contour(xx, yy, zz, linewidths=3, colors=['#a0a0ff', '#0000ff'], levels=[0.002, 0.02])

# 推定された正規分布
zz = multivariate_normal.pdf(np.dstack((xx, yy)), mu_est ,cov_est)
ax.contour(xx, yy, zz, linewidths=3, colors=['#ffa0a0', '#ff0000'], levels=[0.002, 0.02])

ax.axhline(0, color='gray')
ax.axvline(0, color='gray')
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect('equal')
plt.show()

print(f'N = {N}')
print(f'真の平均: {mu} 推定された平均: {mu_est}')
print('真の分散共分散行列')
print(cov)
print('推定された分散共分散行列')
print(cov_est)


---
### $2.$ 確率密度推定の応用

確率密度推定の応用事例を示します．

データに正規分布を当てはめて，この正規分布に対する個々のデータ点の尤度やマハラノビス距離を求めると，それらがこの正規分布から生成されたことの尤もらしさを評価できます．あるデータのマハラノビス距離がとても大きい（尤度がとても小さい）場合，そのデータは，通常のデータの分布から外れた「異常な」ものと判断できそうです．このような考えに基づいて，次のような手順で「異常検出」/「異常検知」（anomaly detection）の仕組みを作ることができます．

1. データを集める．例えば，多数の人々の医療検査のデータ，稼働中の工業機械のデータ（様々な場所の温度，圧力，回転数，etc.）など．
1. 集めたデータに正規分布を当てはめる．
1. 正常/異常を判断するためのマハラノビス距離のしきい値を定める．
1. 新しいデータのマハラノビス距離を測り，その値がしきい値より小さければ正常，大きければ異常と判断する

#### 画像の顔らしさの数値化

「異常検出」とは少し異なりますが，顔の画像をたくさん集めて正規分布を当てはめ，マハラノビス距離によって画像の「顔らしさ」を測る実験をやってみましょう．次のような手順です．

1. 顔画像の位置や大きさを揃えて $64\times 64$ 画素にしたものを大量に用意する．ひとつの顔画像のデータを $\pmb{x}_n$ （$4096\times 1$ 行列，$n = 1, 2, \ldots, N$）と表す（グレイスケール画像なのでデータの次元数は $64\times 64 = 4096$）．
1. 次元削減のため $\{ \pmb{x}_n \}$ に主成分分析を適用．累積寄与率 0.99 以上となる最小の次元数は $700$ だったため，$\{ \pmb{x}_n \}$ の分散共分散行列の大きい方の固有値 $\lambda_1, \lambda_2, \ldots \lambda_{700}$ に対応する固有ベクトルをこの順にならべた $4096\times 700$ の行列 $U$ をつくる．
1. 次元削減したデータ $\pmb{y}_n = U^{\top}(\pmb{x}_n - \pmb{\mu})$ （$\pmb{\mu}$ は $\{ \pmb{x}_n \}$ の平均）の平均は $\pmb{0}$ で分散共分散行列は $\textrm{diag}(\lambda_1, \lambda_2, \ldots, \lambda_{700})$ なので，これらのデータが正規分布に従うと仮定すると，この正規分布に対するマハラノビス距離の2乗は
$$
d^2(\pmb{y}) = \pmb{y}^{\top}
\begin{pmatrix}
\lambda_1 & & 0\\
& \ddots & \\
0 & & \lambda_{700}
\end{pmatrix}\pmb{y} = \frac{y_1^2}{\lambda_1} + \frac{y_2^2}{\lambda_2} + \cdots + \frac{y_{700}^2}{\lambda_{700}}
$$
となる．
1. 画像 $\pmb{x}$ が与えられたら，$\pmb{y} = U^{\top}(\pmb{x} - \pmb{\mu})$ から $d(\pmb{y})$ を求め，その値をその画像の「顔らしさ」とする．

上記のうち 1. と 2. の手順はあらかじめ行ってあります．以下では，そこで得られた $\pmb{\mu}, U, \lambda_1, \ldots, \lambda_{700}$ をファイルから読み込んで，4. の計算を行うようになっています．

この実験では，[LFW (Labeled Faces in the Wild)](https://vis-www.cs.umass.edu/lfw/) という顔画像データセットを用いています．およそ1.3万枚の顔画像から成っています．

In [ ]:
# takataka のウェブサイトからデータを入手
! wget -nc https://www-tlab.math.ryukoku.ac.jp/~takataka/course/MVA/faciallikelihood.npz
import os
path = 'faciallikelihood.npz'
if os.path.exists(path):
    FL = np.load(path)
else:
    print(f'ファイル {path} の読み込みに失敗したようです．再実行してみてください')

In [ ]:
# PCA のパラメータ（平均，固有ベクトル，固有値）
mu, U, lam = FL['mu'], FL['U'], FL['lam']
print(f'mu.shape= {mu.shape}')
print(f'U.shape = {U.shape}')
print(f'lam.shape = {lam.shape}')

次のセルを実行すると，様々なサンプル画像に対して顔らしさを計算し，それらを表示します．

In [ ]:
# サンプル画像
samples = FL['samples']
X = samples.reshape((samples.shape[0], -1)) / 255
# 主成分分析による次元削減
Y = (X - mu) @ U.T
# 推定された正規分布に対するマハラノビス距離の計算
dM = np.sqrt(np.sum((Y**2 / lam), axis=1))
# 画像とそのマハラノビス距離の表示
fig, ax = plt.subplots(3, 6, figsize=(6, 4))
for i in range(3):
    for j in range(6):
        k = i*6 + j
        ax[i, j].imshow(samples[k], cmap='gray', vmin=0, vmax=255)
        ax[i, j].axis('off')
        ax[i, j].set_title(f'{dM[k]:.1f}', fontsize=16)
fig.tight_layout()
plt.show()

左上の画像は，パラメータの推定に用いたデータの平均 $\pmb{\mu}$ です．定義から明らかなように，この画像のマハラノビス距離は $0$ です．

左上の画像以外の17枚の画像は，パラメータの推定に用いたデータの一部を，マハラノビス距離の小さい方から順にならべたものです．
画像を見ると分かるように，「顔らしさ」とは言いつつも，顔の向きが正面でない，濃い陰がある，メガネやゴーグルをかけている等の，顔そのもののつくりとは異なる要素のために顔らしさの数値が小さくなっているようです．

次に，ヒトではないものの顔らしさも求めてみましょう．

In [ ]:
# 画像とそのマハラノビス距離の表示
fig, ax = plt.subplots(1, 6, figsize=(6, 3))
for i in range(6):
    k = i + 18
    ax[i].imshow(samples[k], cmap='gray', vmin=0, vmax=255)
    ax[i].axis('off')
    ax[i].set_title(f'{dM[k]:.1f}', fontsize=16)
fig.tight_layout()
plt.show()



これらは左から，MVAのロゴ，ネコの顔画像3枚，顔のイラスト（「いらすとや」の素材を利用させていただきました），アルチンボルド作の絵画（注）です．

見ての通り，この実験ではあまり良い結果は得られていません．画素値をそのまま用いる方法であるため，顔そのもののつくりを表す特徴と，顔の向きや陰影，アクセサリ等の特徴を区別できていないことが原因と考えられます．機械学習の技術を用いて顔本来の特徴をうまく抽出できるようにすると，改善できるかもしれません．その辺りのことは，学部3年の「機械学習I/II」で少しだけ，大学院の「機械学習特論I/II」などでしっかり学べるかも．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※ 注意: <a href="https://ja.wikipedia.org/wiki/%E3%82%B8%E3%83%A5%E3%82%BC%E3%83%83%E3%83%9A%E3%83%BB%E3%82%A2%E3%83%AB%E3%83%81%E3%83%B3%E3%83%9C%E3%83%AB%E3%83%89">アルチンボルド</a> の「<a href="https://ja.wikipedia.org/wiki/%E3%82%A6%E3%82%A7%E3%83%AB%E3%83%88%E3%82%A5%E3%83%A0%E3%83%8C%E3%82%B9%E3%81%A8%E3%81%97%E3%81%A6%E3%81%AE%E7%9A%87%E5%B8%9D%E3%83%AB%E3%83%89%E3%83%AB%E3%83%952%E4%B8%96%E5%83%8F">ウェルトゥムヌスとしての皇帝ルドルフ2世像</a>」．アルチンボルドは，野菜・果物や花などを組み合わせた肖像画でよく知られた16世紀の画家．
</span>

#### 自前の画像でもやってみる？

以下では，自分で用意した顔画像で顔らしさを計算できます．

In [ ]:
# 顔検出器の準備
faceCascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')

# Colab へのファイルアップロードを実行する関数
#
def uploadToColab():
    try:
        from google.colab import files
        rv = files.upload()
    except:
        print('このコードは Colab 以外の環境では実行できないよ．')

# OpenCVの形式の画像を表示する関数
#
def imshow(img):
    try:
        from google.colab.patches import cv2_imshow
        cv2_imshow(img) # Colab上で実行している場合
    except:
        cv2.imshow(img)  # それ以外の場合

# 与えられた画像の中から顔を検出する
#
def faceDetector(img, faceCascade, maxSize=400):

    h, w = img.shape[:2]

    # (短辺の長さ) <= maxSize にリサイズ
    if min(w, h) > maxSize:
        if w <= h:
            w2, h2 = maxSize, h*maxSize//w
        else:
            w2, h2 = w*maxSize//h, maxSize
        imgDisp = cv2.resize(img, (w2, h2))
    else:
        imgDisp = np.copy(img)

    # 顔検出を実行
    imgGray = cv2.cvtColor(imgDisp, cv2.COLOR_BGR2GRAY)
    faces = faceCascade.detectMultiScale(imgGray, 1.05, 4)
    nFace = len(faces)

    # 検出した顔のうち最初のひとりを選んで切り取り
    if nFace > 0:
        print(faces)
        x, y, ww, hh = faces[0]
        imgFace = np.copy(imgDisp[y:y+hh, x:x+ww, :])
        cv2.rectangle(imgDisp, (x, y), (x+ww, y+hh), color=(0, 255, 0), thickness=2)
        return nFace, imgDisp, imgFace
    else:
        return nFace, imgDisp, None


(1) 画像を Colab へアップロードします．次のような画像にしてください

- ひとりのひとのほぼ正面を向いた顔全体が写ってる．複数人を検出した場合，ひとりだけ抜き出します（あとで条件に合わない画像をわざと与えて実験してみるのもよいでしょう）．
- 画像全体に顔がドアップで写っているような場合はうまく顔検出ができないかもしれません．顔がもう少し小さく写ってる画像を探してみましょう．
- ファイル名に空白やマルチバイト文字（日本語など）が含まれているとうまく動作しない場合があるので，自分のPCの方で適当な名前に変えておく（拡張子は変えてはいけない）．

In [ ]:
# Colab へファイルをアップロード
uploadToColab()

# ls コマンドでファイルを一覧
! ls

上のセルを実行してファイルをアップロードできたら，次のセルのファイル名の部分にその名前を指定して，読み込ませましょう．

In [ ]:
# 画像を読み込む．`hoge.jpg` を自分がアップロードしたファイルの名前に修正
img = cv2.imread('hoge.png')
if img is None:
    print('画像を読み込めませんでした．やり直してください')

(2) 顔検出を実行して顔領域を切り取った画像を作る

In [ ]:
# 顔を検出する
numFace, imgDisp, imgFace = faceDetector(img, faceCascade, maxSize=400)
if numFace > 0:
    imshow(imgDisp)
    imshow(imgFace)
else:
    imshow(imgDisp)
    print('顔を検出できませんでした')

(3) 顔を所定の大きさにリサイズしてから次元削減し，マハラノビス距離を計算．

In [ ]:
# 使用する画像の選択
imgInput = imgFace  # 顔検出して切り取ったものを使用
#imgInput = img     # 読み込んだ画像をそのまま使用

# 顔画像を 64 x 64 にリサイズ
imgInput2 = cv2.resize(cv2.cvtColor(imgInput, cv2.COLOR_BGR2GRAY), (64, 64))
# 4096 次元ベクトルにする
xvec = imgInput2.reshape(-1) / 255
# PCAを利用した次元削減
yvec = (xvec - mu) @ U.T
# マハラノビス距離の計算
dM = np.sqrt(np.sum((yvec**2 / lam)))
# 画像を表示
imshow(imgInput2)
# 結果の表示
print(f'顔らしさ {dM:.2f}')

上記の実験では，顔検出を経由するため，顔検出ができないような画像で試すことができません．顔でない画像で試したいひとは，次のように実行してみてね．

1. (1) を普通に実行
1. (2) は実行しない
1. (3) のコードセル上部を次のように修正（`#`の場所を変える）して実行する
```
#imgInput = imgFace  # 顔検出して切り取ったものを使用
imgInput = img     # 読み込んだ画像をそのまま使用
```

ただし，入力した画像全体を 64 x 64 に縮小して用いることになります．画像の一部を使い対場合は，アップロードする前に手元で画像を加工するか，画像の一部を切り取るコードを追加する必要があります．後者を試したいひとは takataka に相談してね．